In [1]:
# libraries — run after 01_import; no dependence on old 02/03 outputs
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd


In [2]:
# globals and traceable inputs; works from project root or b_simulate/src
ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent.parent
for task, target in [('b_simulate','../../a_import/output'), ('c_analyze','../../b_simulate/output')]:
    folder = ROOT/task/'input'
    folder.mkdir(parents=True, exist_ok=True)
    link = folder/'upstream'
    link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)
cfg = json.loads((ROOT/'b_simulate/input/upstream/config.json').read_text())
SEED = cfg['seed']
OUT = ROOT/'b_simulate/output/varying_wind'
OUT.mkdir(parents=True, exist_ok=True)


In [3]:
# variable block lengths — same mean as fixed L=16; bounded moderate variability
# population law Uniform{4,...,28}: mean=16, variance=52, sd=7.21, CV=0.45
# matched signs and balanced records induce finite-record dependence; lengths are not iid
settings = {'fixed_blocks':[1,16,32], 'variable_mean':16, 'variable_min':4,
    'variable_max':28, 'bootstrap_draws':2000, 'wind_max_lag':96, 'gif_fps':10,
    'display_halfwidth':48}
conditions = [('fixed_1','Fixed L=1',1,False), ('fixed_16','Fixed L=16',16,False),
    ('fixed_32','Fixed L=32',32,False), ('variable_16','Variable L=4–28',16,True)]
support = np.arange(settings['variable_min'], settings['variable_max']+1)
design = pd.DataFrame([
    {'condition':key, 'label':label, 'mean_block':length,
     'population_block_variance':support.var() if variable else 0.,
     'population_block_sd':support.std() if variable else 0., 'variable':variable}
    for key,label,length,variable in conditions])
design.to_csv(OUT/'design.csv', index=False)
design


,condition,label,mean_block,population_block_variance,population_block_sd,variable
0,fixed_1,Fixed L=1,1,0.0,0.000000,False
1,fixed_16,Fixed L=16,16,0.0,0.000000,False
2,fixed_32,Fixed L=32,32,0.0,0.000000,False
3,variable_16,Variable L=4–28,16,52.0,7.211103,True


In [4]:
# balanced wind schedules; fixed L=16 and variable case share sign order and phase per trial
# +/- durations use the same multiset, shuffled separately: equal time and duration histograms
def make_wind(cfg, settings, length, variable):
    n_steps = cfg['n_steps']
    n_blocks = n_steps//length
    assert n_steps % (2*length) == 0
    if variable:
        assert n_blocks % 4 == 0
        assert settings['variable_min']+settings['variable_max'] == 2*length
    wind = np.empty((cfg['n_trials'], n_steps), dtype=np.int8)
    blocks = []
    for trial in range(cfg['n_trials']):
        sign_rng = np.random.default_rng(np.random.SeedSequence(SEED, spawn_key=(201,trial)))
        phase_rng = np.random.default_rng(np.random.SeedSequence(SEED, spawn_key=(202,trial)))
        length_rng = np.random.default_rng(np.random.SeedSequence(SEED, spawn_key=(203,trial)))
        signs = sign_rng.permutation(np.repeat([-1,1], n_blocks//2))
        durations = np.full(n_blocks, length, dtype=int)
        if variable:
            draws = length_rng.integers(settings['variable_min'], settings['variable_max']+1, size=n_blocks//4)
            bag = np.r_[draws, 2*length-draws]
            durations[signs==1] = length_rng.permutation(bag)
            durations[signs==-1] = length_rng.permutation(bag)
        phase = int(phase_rng.integers(n_steps))
        wind[trial] = np.roll(np.repeat(signs,durations), -phase)
        blocks.extend({'trial':trial,'block':i,'sign':int(sign),'length':int(duration),'phase':phase}
            for i,(sign,duration) in enumerate(zip(signs,durations)))
    # periodic continuation into burn-in; all trials start at random phases of their record
    startup = wind[:, np.arange(-cfg['burn_in'],0) % n_steps]
    return np.concatenate([startup,wind], axis=1), pd.DataFrame(blocks)


In [5]:
# baseline kernel is used only for established-plume initialization
def baseline_kernel(cfg):
    kernel = np.zeros((cfg['nx'],cfg['ny']))
    kernel[0,cfg['source_y']] = 1
    d = cfg['diffusivity']
    for x in range(1,cfg['nx']):
        kernel[x] = (1-2*d)*kernel[x-1]
        kernel[x,1:] += d*kernel[x-1,:-1]
        kernel[x,:-1] += d*kernel[x-1,1:]
    return kernel

def initial_grid(rng,cfg):
    counts = np.zeros((cfg['n_trials'],cfg['nx'],cfg['ny']),dtype=np.int64)
    initial = np.array(['clean']*(cfg['n_trials']//2)+['established']*(cfg['n_trials']//2))
    kernel = baseline_kernel(cfg)
    for x in range(cfg['nx']):
        counts[cfg['n_trials']//2:,x] = rng.multinomial(
            cfg['packets_per_step'],kernel[x],size=cfg['n_trials']//2)
    return counts,initial


In [6]:
# local synchronous CA — sample only occupied cells; empty cells contribute no transfers
# indices within a directional transfer are unique, so sequential += correctly combines arrivals
def step_ca(counts,sign,rng,cfg):
    old = counts[:,:-1]
    trial,x,y = np.nonzero(old)
    occupied = old[trial,x,y]
    p_up = cfg['diffusivity']+cfg['wind_bias']*sign[trial]/2
    p_down = cfg['diffusivity']-cfg['wind_bias']*sign[trial]/2
    up = rng.binomial(occupied,p_up)
    down = rng.binomial(occupied-up,p_down/(1-p_up))
    new = np.zeros_like(counts)
    new[trial,x+1,y] += occupied-up-down
    keep_up = y < cfg['ny']-1
    keep_down = y > 0
    new[trial[keep_up],x[keep_up]+1,y[keep_up]+1] += up[keep_up]
    new[trial[keep_down],x[keep_down]+1,y[keep_down]-1] += down[keep_down]
    lateral = np.bincount(trial[~keep_up],weights=up[~keep_up],minlength=cfg['n_trials'])
    lateral += np.bincount(trial[~keep_down],weights=down[~keep_down],minlength=cfg['n_trials'])
    lost = counts[:,-1].sum(axis=1)+lateral.astype(np.int64)
    new[:,0,cfg['source_y']] = cfg['packets_per_step']
    return new,lost,lateral.astype(np.int64)


In [7]:
# execute one condition; store actual snapshot times to keep GIF indexing aligned
def simulate_condition(cfg,settings,condition_index,length,variable):
    packet_rng = np.random.default_rng(np.random.SeedSequence(SEED,spawn_key=(210,condition_index)))
    wind,blocks = make_wind(cfg,settings,length,variable)
    counts,initial = initial_grid(packet_rng,cfg)
    total = wind.shape[1]
    trace = np.zeros((cfg['n_trials'],total,len(cfg['receptor_offsets'])),dtype=np.float32)
    mean_grid = np.zeros_like(counts,dtype=np.float32)
    frames,frame_steps = [],[]
    previous_mass = counts.sum(axis=(1,2))
    error = np.zeros(cfg['n_trials'],dtype=np.int64)
    lateral_total = np.zeros(cfg['n_trials'],dtype=np.int64)
    for t in range(total):
        counts,lost,lateral = step_ca(counts,wind[:,t],packet_rng,cfg)
        mass = counts.sum(axis=(1,2))
        error = np.maximum(error,np.abs(mass-previous_mass-cfg['packets_per_step']+lost))
        previous_mass = mass
        lateral_total += lateral
        if t >= cfg['burn_in']:
            mean_grid += counts
        for j,offset in enumerate(cfg['receptor_offsets']):
            center = cfg['source_y']+offset
            h = cfg['receptor_halfwidth']
            trace[:,t,j] = counts[:,cfg['receptor_x'],center-h:center+h+1].mean(axis=1)
        if t % cfg['frame_interval'] == 0 and len(frames) < cfg['demo_frames']:
            frames.append(counts[0].astype(np.uint8))
            frame_steps.append(t+1)
    audit = pd.DataFrame({'trial':np.arange(cfg['n_trials']),'initial':initial,
        'max_mass_error':error,'lateral_loss':lateral_total,
        'positive_fraction':(wind[:,cfg['burn_in']:]==1).mean(axis=1)})
    return {'mean_grid':mean_grid/cfg['n_steps'],'trace':trace,'wind':wind,
        'frames':np.stack(frames),'frame_steps':np.array(frame_steps),'initial':initial},blocks,audit


In [8]:
# run all conditions — 64 trials each; every output below is generated by this notebook
all_blocks,all_audits,runtimes = [],[],[]
for index,(key,label,length,variable) in enumerate(conditions):
    started = time.perf_counter()
    result,blocks,audit = simulate_condition(cfg,settings,index,length,variable)
    np.savez_compressed(OUT/f'{key}.npz',**result)
    all_blocks.append(blocks.assign(condition=key))
    all_audits.append(audit.assign(condition=key))
    runtimes.append({'condition':label,'seconds':time.perf_counter()-started,
        'snapshots':len(result['frame_steps']),'last_snapshot_step':result['frame_steps'][-1]})
blocks = pd.concat(all_blocks,ignore_index=True)
audit = pd.concat(all_audits,ignore_index=True)
blocks.to_csv(OUT/'block_lengths.csv',index=False)
audit.to_csv(OUT/'simulation_audit.csv',index=False)
pd.DataFrame(runtimes).to_csv(OUT/'runtimes.csv',index=False)
(OUT/'config.json').write_text(json.dumps(cfg,indent=2))
(OUT/'varying_wind_settings.json').write_text(json.dumps(settings,indent=2))
pd.DataFrame(runtimes)


,condition,seconds,snapshots,last_snapshot_step
0,Fixed L=1,59.610324,144,1145
1,Fixed L=16,66.702506,144,1145
2,Fixed L=32,68.593620,144,1145
3,Variable L=4–28,67.521992,144,1145


In [9]:
# transport and experimental-control checks, performed where the data are generated
assert audit['max_mass_error'].max() == 0
assert (audit['positive_fraction']==.5).all()
generated = blocks.groupby(['condition','trial','sign'])['length'].agg(['mean','sum'])
assert (generated['sum']==cfg['n_steps']//2).all()
assert (generated.loc['variable_16','mean']==settings['variable_mean']).all()
audit.groupby('condition',sort=False).agg(trials=('trial','size'),
    max_mass_error=('max_mass_error','max'),lateral_loss=('lateral_loss','sum'),
    positive_fraction=('positive_fraction','mean'))


,trials,max_mass_error,lateral_loss,positive_fraction
condition,,,,
fixed_1,64,0,0,0.5
fixed_16,64,0,0,0.5
fixed_32,64,0,0,0.5
variable_16,64,0,0,0.5


In [10]:
# methods notes; L retains its original meaning as a block length
methods = pd.DataFrame([
    ['question','Does variation in block length change exposure when mean block length is held at 16?'],
    ['reference','Fixed L=16 is the matched-mean reference; L=1 and L=32 provide context'],
    ['variable law','Each generated block has marginal Uniform{4,...,28}; mean 16, variance 52'],
    ['balance','Durations are paired around 16; each sign receives the same multiset, separately shuffled'],
    ['dependence','Balancing couples block lengths within each finite record; they are not iid'],
    ['matched wind order','Fixed L=16 and variable case share sign order and circular phase within each trial'],
    ['wind runs','Adjacent blocks can share a sign; actual uninterrupted winds can last multiple blocks'],
    ['transport','Original east/NE/SE CA; same source, wind bias, diffusion parameter, grid, and receptors'],
    ['initialization','Half clean, half established; main statistics exclude burn-in; GIF shows first clean trial'],
    ['scope','Illustrative duration distribution, not calibrated weather; all units are cells and steps'],
    ['replication','All comparison conditions are regenerated here; no old simulation NPZ files are required']
],columns=['item','specification'])
methods.to_csv(OUT/'methods.csv',index=False)
methods


,item,specification
0,question,Does variation in block length change exposure...
1,reference,Fixed L=16 is the matched-mean reference; L=1 ...
2,variable law,"Each generated block has marginal Uniform{4,....."
3,balance,Durations are paired around 16; each sign rece...
4,dependence,Balancing couples block lengths within each fi...
5,matched wind order,Fixed L=16 and variable case share sign order ...
6,wind runs,Adjacent blocks can share a sign; actual unint...
7,transport,"Original east/NE/SE CA; same source, wind bias..."
8,initialization,"Half clean, half established; main statistics ..."
9,scope,"Illustrative duration distribution, not calibr..."
